# Interactive 2D curve morphing demo (Bézier / B-spline).



Install:
    python -m pip install numpy

Run as a standalone desktop program:
    python curve_morphing_interactive.py

How to use:
- Choose curve A or B in the left panel.
- Choose Bézier or B-spline for each curve.
- Click in the canvas to add a control point to the active curve.
- Drag any existing control point to move it (including points on the other curve).
- Use Undo last point / Clear active curve as needed.
- Move the progress slider, or use Play/Pause to animate the morph.

The animation is driven by Tkinter's event loop, not Matplotlib's animation timer.
The morph itself is sample-based: evaluate each curve at the same normalized t values,
then interpolate corresponding samples. This accommodates different control-point counts.


## 1. Imports and general settings

In [5]:
# -----------------------------------------------------------------------------

from __future__ import annotations

from math import comb
import tkinter as tk
from tkinter import messagebox, ttk
from typing import Literal

import numpy as np

CurveType = Literal["bezier", "bspline"]

SAMPLE_COUNT = 240          # Number of points used to approximate each curve
ANIMATION_STEP = 0.008      # Increase in morphing progress per animation frame
ANIMATION_DELAY_MS = 30     # Time between animation frames


# -----------------------------------------------------------------------------


## 2. Bézier and B-spline curve evaluation

In [6]:
def evaluate_bezier(
    control_points: np.ndarray,
    t_values: np.ndarray,
) -> np.ndarray:
    """Evaluate a Bézier curve using the Bernstein-polynomial formula."""
    points = np.asarray(control_points, dtype=float)
    ts = np.asarray(t_values, dtype=float)

    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("Control points must be an array of 2D points.")
    if len(points) < 2:
        return np.empty((0, 2), dtype=float)

    degree = len(points) - 1
    curve_points = np.zeros((len(ts), 2), dtype=float)

    # Each control point contributes according to its Bernstein basis function.
    for i, point in enumerate(points):
        weight = comb(degree, i) * (1.0 - ts) ** (degree - i) * ts**i
        curve_points += weight[:, None] * point[None, :]

    return curve_points

def create_open_uniform_knot_vector(
    number_of_controls: int,
    degree: int,
) -> np.ndarray:
    """Create an open-uniform knot vector whose parameter domain is [0, 1]."""
    if degree < 1:
        raise ValueError("B-spline degree must be at least 1.")
    if number_of_controls < degree + 1:
        raise ValueError("A B-spline needs at least degree + 1 control points.")

    # Open (clamped) knots repeat each end knot degree + 1 times.
    number_of_internal_knots = number_of_controls - degree - 1
    if number_of_internal_knots > 0:
        internal_knots = np.linspace(
            0.0, 1.0, number_of_internal_knots + 2
        )[1:-1]
    else:
        internal_knots = np.array([], dtype=float)

    return np.concatenate(
        (
            np.zeros(degree + 1),
            internal_knots,
            np.ones(degree + 1),
        )
    )

def bspline_basis(
    index: int,
    degree: int,
    t: float,
    knots: np.ndarray,
) -> float:
    """Evaluate one B-spline basis function using Cox-de Boor recursion."""
    # Degree-zero basis functions are 1 on their knot interval and 0 elsewhere.
    if degree == 0:
        return float(knots[index] <= t < knots[index + 1])

    left_denominator = knots[index + degree] - knots[index]
    right_denominator = knots[index + degree + 1] - knots[index + 1]

    left_term = 0.0
    right_term = 0.0

    # A zero denominator occurs with repeated knots; that term contributes zero.
    if left_denominator > 0.0:
        left_term = (
            (t - knots[index])
            / left_denominator
            * bspline_basis(index, degree - 1, t, knots)
        )

    if right_denominator > 0.0:
        right_term = (
            (knots[index + degree + 1] - t)
            / right_denominator
            * bspline_basis(index + 1, degree - 1, t, knots)
        )

    return left_term + right_term

def evaluate_bspline(
    control_points: np.ndarray,
    t_values: np.ndarray,
    degree: int = 3,
) -> np.ndarray:
    """Evaluate an open-uniform B-spline over the normalized interval [0, 1]."""
    points = np.asarray(control_points, dtype=float)
    ts = np.asarray(t_values, dtype=float)

    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("Control points must be an array of 2D points.")
    if len(points) < 2:
        return np.empty((0, 2), dtype=float)

    # Use a lower degree while the user is still entering fewer than four points.
    actual_degree = min(degree, len(points) - 1)
    knots = create_open_uniform_knot_vector(len(points), actual_degree)
    curve_points = np.zeros((len(ts), 2), dtype=float)
    domain_end = knots[-actual_degree - 1]

    for sample_index, t in enumerate(ts):
        if np.isclose(t, domain_end):
            # The degree-zero definition uses half-open intervals, so t = 1
            # needs to be handled explicitly for a clamped curve.
            curve_points[sample_index] = points[-1]
            continue

        for i, control_point in enumerate(points):
            weight = bspline_basis(i, actual_degree, float(t), knots)
            curve_points[sample_index] += weight * control_point

    return curve_points

def evaluate_curve(
    control_points: list[list[float]] | np.ndarray,
    curve_type: CurveType,
    t_values: np.ndarray,
) -> np.ndarray:
    """Evaluate the selected curve type using its own mathematical definition."""
    points = np.asarray(control_points, dtype=float)

    if len(points) < 2:
        return np.empty((0, 2), dtype=float)
    if curve_type == "bezier":
        return evaluate_bezier(points, t_values)
    if curve_type == "bspline":
        return evaluate_bspline(points, t_values, degree=3)

    raise ValueError(f"Unsupported curve type: {curve_type}")


## 3. Graphical application

In [7]:
class CurveMorphingApp:
    """Tkinter interface for creating, editing and morphing two planar curves."""

    # Coordinates used internally for the drawing area.
    WORLD_X_MIN, WORLD_X_MAX = 0.0, 10.0
    WORLD_Y_MIN, WORLD_Y_MAX = 0.0, 7.0

    BLUE = "#2878c8"
    ORANGE = "#d9822b"
    GREEN = "#219653"
    GRID = "#e8edf2"
    TEXT = "#243447"

    def __init__(self, root: tk.Tk) -> None:
        self.root = root
        self.root.title("2D Curve Morphing — Bézier and B-spline")
        self.root.geometry("1220x790")
        self.root.minsize(940, 620)

        # Initial examples: A is cubic Bézier (4 controls), B is cubic B-spline (6).
        self.controls: dict[str, list[list[float]]] = {
            "A": [[0.8, 0.8], [1.0, 5.7], [3.4, 5.6], [4.4, 0.9]],
            "B": [
                [1.0, 1.1], [0.7, 3.8], [2.0, 6.2],
                [4.1, 5.1], [5.6, 1.0], [8.8, 3.3],
            ],
        }
        self.curve_types: dict[str, CurveType] = {
            "A": "bezier",
            "B": "bspline",
        }

        # Shared parameter values make it possible to pair samples from A and B.
        self.t_values = np.linspace(0.0, 1.0, SAMPLE_COUNT)
        self.progress = 0.0
        self.active_curve = "A"
        self.dragging: tuple[str, int] | None = None
        self.playing = False
        self.after_id: str | None = None

        # Tkinter variables keep the controls in the interface synchronized.
        self.active_var = tk.StringVar(value="A")
        self.type_a_var = tk.StringVar(value="bezier")
        self.type_b_var = tk.StringVar(value="bspline")
        self.progress_var = tk.DoubleVar(value=0.0)
        self.status_var = tk.StringVar(value="Select a curve and click in the drawing area to add points.")

        self.build_layout()
        self.canvas.bind("<Configure>", self.on_canvas_resize)
        self.canvas.bind("<Button-1>", self.on_canvas_press)
        self.canvas.bind("<B1-Motion>", self.on_canvas_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_canvas_release)

        self.redraw()
        self.root.after(700, self.start_if_ready)
        self.root.protocol("WM_DELETE_WINDOW", self.close)

    def build_layout(self) -> None:
        """Create the side panel, editing controls and drawing canvas."""
        outer = ttk.Frame(self.root, padding=10)
        outer.pack(fill="both", expand=True)

        side = ttk.Frame(outer, width=265)
        side.pack(side="left", fill="y", padx=(0, 12))
        side.pack_propagate(False)

        ttk.Label(
            side,
            text="2D Curve Morphing",
            font=("TkDefaultFont", 15, "bold"),
        ).pack(anchor="w", pady=(0, 4))
        ttk.Label(
            side,
            text="Create two curves, edit their control points, and transform one into the other.",
            wraplength=245,
            justify="left",
        ).pack(anchor="w", pady=(0, 12))

        # Select which curve receives newly added points.
        active_box = ttk.LabelFrame(side, text="Curve to add points to", padding=8)
        active_box.pack(fill="x", pady=(0, 10))
        ttk.Radiobutton(
            active_box, text="Curve A", variable=self.active_var,
            value="A", command=self.on_active_change,
        ).pack(anchor="w")
        ttk.Radiobutton(
            active_box, text="Curve B", variable=self.active_var,
            value="B", command=self.on_active_change,
        ).pack(anchor="w")

        # Select Bézier or B-spline independently for each curve.
        types_box = ttk.LabelFrame(side, text="Curve types", padding=8)
        types_box.pack(fill="x", pady=(0, 10))
        ttk.Label(types_box, text="Curve A").grid(row=0, column=0, sticky="w", pady=3)
        type_a = ttk.Combobox(
            types_box, textvariable=self.type_a_var,
            values=("bezier", "bspline"), state="readonly", width=11,
        )
        type_a.grid(row=0, column=1, sticky="e", pady=3)
        type_a.bind(
            "<<ComboboxSelected>>",
            lambda _event: self.on_type_change("A", self.type_a_var.get()),
        )

        ttk.Label(types_box, text="Curve B").grid(row=1, column=0, sticky="w", pady=3)
        type_b = ttk.Combobox(
            types_box, textvariable=self.type_b_var,
            values=("bezier", "bspline"), state="readonly", width=11,
        )
        type_b.grid(row=1, column=1, sticky="e", pady=3)
        type_b.bind(
            "<<ComboboxSelected>>",
            lambda _event: self.on_type_change("B", self.type_b_var.get()),
        )

        # Buttons for editing the active curve's control points.
        edit_box = ttk.LabelFrame(side, text="Edit control points", padding=8)
        edit_box.pack(fill="x", pady=(0, 10))
        ttk.Button(edit_box, text="Undo last point", command=self.undo_point).pack(fill="x", pady=3)
        ttk.Button(edit_box, text="Clear active curve", command=self.clear_active).pack(fill="x", pady=3)
        ttk.Button(edit_box, text="Reset example curves", command=self.reset_examples).pack(fill="x", pady=3)

        # Controls for the morphing parameter s.
        morph_box = ttk.LabelFrame(side, text="Morphing", padding=8)
        morph_box.pack(fill="x", pady=(0, 10))
        ttk.Label(morph_box, text="Transformation progress").pack(anchor="w")
        self.progress_scale = tk.Scale(
            morph_box,
            variable=self.progress_var,
            from_=0.0,
            to=1.0,
            resolution=0.005,
            orient="horizontal",
            showvalue=True,
            command=self.on_progress_change,
            length=220,
        )
        self.progress_scale.pack(fill="x", pady=(0, 6))
        self.play_button = ttk.Button(morph_box, text="Play", command=self.toggle_play)
        self.play_button.pack(fill="x", pady=3)
        ttk.Button(morph_box, text="Back to 0%", command=self.reset_progress).pack(fill="x", pady=3)

        help_box = ttk.LabelFrame(side, text="How to use", padding=8)
        help_box.pack(fill="x", pady=(0, 10))
        help_text = (
            "• Click empty canvas space to add a point to the selected curve.\n"
            "• Drag any control point to move it.\n"
            "• A Bézier curve uses all its control points.\n"
            "• A B-spline becomes cubic once it has 4+ points.\n"
            "• The two curves may have different point counts."
        )
        ttk.Label(help_box, text=help_text, wraplength=240, justify="left").pack(anchor="w")

        ttk.Label(
            side,
            textvariable=self.status_var,
            wraplength=245,
            justify="left",
            foreground="#40566f",
        ).pack(anchor="w", pady=(4, 0))

        self.canvas = tk.Canvas(
            outer, background="white", highlightthickness=1,
            highlightbackground="#cbd5e1",
        )
        self.canvas.pack(side="left", fill="both", expand=True)


# -----------------------------------------------------------------------------
# 4. Coordinate conversion and drawing
# -----------------------------------------------------------------------------

    def plot_geometry(self) -> tuple[float, float, float]:
        """Return the canvas origin and scale used for world-to-screen mapping."""
        width = max(self.canvas.winfo_width(), 200)
        height = max(self.canvas.winfo_height(), 200)
        margin = 42.0
        scale = min(
            (width - 2 * margin) / (self.WORLD_X_MAX - self.WORLD_X_MIN),
            (height - 2 * margin) / (self.WORLD_Y_MAX - self.WORLD_Y_MIN),
        )
        origin_x = (width - (self.WORLD_X_MAX - self.WORLD_X_MIN) * scale) / 2.0
        baseline_y = (height + (self.WORLD_Y_MAX - self.WORLD_Y_MIN) * scale) / 2.0
        return origin_x, baseline_y, scale

    def world_to_screen(self, x: float, y: float) -> tuple[float, float]:
        """Convert mathematical coordinates to Tkinter canvas coordinates."""
        origin_x, baseline_y, scale = self.plot_geometry()
        screen_x = origin_x + (x - self.WORLD_X_MIN) * scale
        screen_y = baseline_y - (y - self.WORLD_Y_MIN) * scale
        return screen_x, screen_y

    def screen_to_world(self, x: float, y: float) -> tuple[float, float]:
        """Convert mouse coordinates back to the mathematical coordinate system."""
        origin_x, baseline_y, scale = self.plot_geometry()
        world_x = (x - origin_x) / scale + self.WORLD_X_MIN
        world_y = (baseline_y - y) / scale + self.WORLD_Y_MIN
        return world_x, world_y

    def inside_plot(self, x: float, y: float) -> bool:
        """Check whether a screen position lies inside the drawing rectangle."""
        world_x, world_y = self.screen_to_world(x, y)
        return (
            self.WORLD_X_MIN <= world_x <= self.WORLD_X_MAX
            and self.WORLD_Y_MIN <= world_y <= self.WORLD_Y_MAX
        )

    @staticmethod
    def as_canvas_coords(points: np.ndarray, convert) -> list[float]:
        """Flatten a sequence of 2D points into the coordinate list Tkinter expects."""
        coordinates: list[float] = []
        for x, y in points:
            screen_x, screen_y = convert(float(x), float(y))
            coordinates.extend((screen_x, screen_y))
        return coordinates

    def draw_grid(self) -> None:
        """Draw the coordinate frame and grid lines."""
        origin_x, baseline_y, scale = self.plot_geometry()
        left = origin_x
        right = origin_x + (self.WORLD_X_MAX - self.WORLD_X_MIN) * scale
        top = baseline_y - (self.WORLD_Y_MAX - self.WORLD_Y_MIN) * scale
        bottom = baseline_y

        self.canvas.create_rectangle(left, top, right, bottom, outline="#aebdca", width=1)

        for x in range(int(self.WORLD_X_MIN), int(self.WORLD_X_MAX) + 1):
            screen_x, _ = self.world_to_screen(float(x), self.WORLD_Y_MIN)
            self.canvas.create_line(screen_x, top, screen_x, bottom, fill=self.GRID)
            self.canvas.create_text(
                screen_x, bottom + 15, text=str(x),
                fill="#65758b", font=("TkDefaultFont", 8),
            )

        for y in range(int(self.WORLD_Y_MIN), int(self.WORLD_Y_MAX) + 1):
            _, screen_y = self.world_to_screen(self.WORLD_X_MIN, float(y))
            self.canvas.create_line(left, screen_y, right, screen_y, fill=self.GRID)
            self.canvas.create_text(
                left - 15, screen_y, text=str(y),
                fill="#65758b", font=("TkDefaultFont", 8),
            )

        self.canvas.create_text((left + right) / 2, bottom + 32, text="x", fill=self.TEXT)
        self.canvas.create_text(left - 28, (top + bottom) / 2, text="y", fill=self.TEXT)

    def get_curve_points(self, which: str) -> np.ndarray:
        """Evaluate curve A or B at the common parameter values."""
        return evaluate_curve(
            self.controls[which],
            self.curve_types[which],
            self.t_values,
        )

    def draw_control_polygon(self, which: str, color: str) -> None:
        """Draw the control polygon and the draggable control points."""
        control_points = self.controls[which]

        if len(control_points) >= 2:
            coordinates = self.as_canvas_coords(
                np.asarray(control_points), self.world_to_screen
            )
            self.canvas.create_line(
                *coordinates, fill=color, width=1.3, dash=(3, 4)
            )

        for index, (x, y) in enumerate(control_points):
            screen_x, screen_y = self.world_to_screen(x, y)
            radius = 6
            outline = color if which != self.active_curve else "#172b4d"
            self.canvas.create_oval(
                screen_x - radius, screen_y - radius,
                screen_x + radius, screen_y + radius,
                fill="white", outline=outline, width=2,
            )
            self.canvas.create_text(
                screen_x + 9, screen_y - 10,
                text=f"{which}{index + 1}", anchor="w",
                fill=color, font=("TkDefaultFont", 8, "bold"),
            )

    def draw_polyline(
        self,
        points: np.ndarray,
        color: str,
        width: int,
        dash=None,
    ) -> None:
        """Draw sampled curve points as connected straight line segments."""
        if len(points) < 2:
            return

        coordinates = self.as_canvas_coords(points, self.world_to_screen)
        options = {
            "fill": color,
            "width": width,
            "joinstyle": "round",
            "capstyle": "round",
        }
        if dash is not None:
            options["dash"] = dash
        self.canvas.create_line(*coordinates, **options)

    def redraw(self) -> None:
        """Recalculate and redraw both curves and the current intermediate shape."""
        if not hasattr(self, "canvas") or not self.canvas.winfo_exists():
            return

        self.canvas.delete("all")
        self.draw_grid()

        curve_a = self.get_curve_points("A")
        curve_b = self.get_curve_points("B")

        # Draw the two original curves as lighter dashed lines.
        if len(curve_a) >= 2:
            self.draw_polyline(curve_a, "#9bbfe5", 2, dash=(5, 4))
        if len(curve_b) >= 2:
            self.draw_polyline(curve_b, "#edbd8b", 2, dash=(5, 4))

        self.draw_control_polygon("A", self.BLUE)
        self.draw_control_polygon("B", self.ORANGE)

        if len(curve_a) >= 2 and len(curve_b) >= 2:
            # Main morphing step: interpolate corresponding sampled positions.
            intermediate = (1.0 - self.progress) * curve_a + self.progress * curve_b
            self.draw_polyline(intermediate, self.GREEN, 3)

            # Mark the two endpoints of the current intermediate curve.
            for point in (intermediate[0], intermediate[-1]):
                screen_x, screen_y = self.world_to_screen(
                    float(point[0]), float(point[1])
                )
                self.canvas.create_oval(
                    screen_x - 4, screen_y - 4,
                    screen_x + 4, screen_y + 4,
                    fill=self.GREEN, outline="white", width=1,
                )
        else:
            self.canvas.create_text(
                self.canvas.winfo_width() / 2,
                self.canvas.winfo_height() / 2,
                text="Add at least 2 control points to each curve to see the morph.",
                fill=self.TEXT,
                font=("TkDefaultFont", 12),
            )

        self.canvas.create_text(
            18, 16, anchor="w",
            text="Blue: curve A     Orange: curve B     Green: intermediate curve",
            fill=self.TEXT, font=("TkDefaultFont", 10, "bold"),
        )
        self.update_status()

    def update_status(self) -> None:
        """Refresh the text showing the current curve settings and morph progress."""
        count_a = len(self.controls["A"])
        count_b = len(self.controls["B"])
        self.status_var.set(
            f"Editing: curve {self.active_curve}\n"
            f"Curve A: {self.curve_types['A']} ({count_a} control points)\n"
            f"Curve B: {self.curve_types['B']} ({count_b} control points)\n"
            f"Morph: {self.progress:.0%}\n\n"
            "The curves may have different numbers of control points."
        )


# -----------------------------------------------------------------------------
# 5. Mouse and editing events
# -----------------------------------------------------------------------------

    def on_active_change(self) -> None:
        self.active_curve = self.active_var.get()
        self.update_status()
        self.redraw()

    def on_type_change(self, which: str, curve_type: str) -> None:
        self.curve_types[which] = curve_type  # Selected from the two allowed options.
        self.update_status()
        self.redraw()

    def on_canvas_resize(self, _event=None) -> None:
        self.redraw()

    def on_canvas_press(self, event) -> None:
        """Select a nearby control point or add a new point to the active curve."""
        nearest: tuple[float, str, int] | None = None

        # Check control points from both curves, not only the active one.
        for which in ("A", "B"):
            for index, (x, y) in enumerate(self.controls[which]):
                screen_x, screen_y = self.world_to_screen(x, y)
                distance = ((screen_x - event.x) ** 2 + (screen_y - event.y) ** 2) ** 0.5
                if nearest is None or distance < nearest[0]:
                    nearest = (distance, which, index)

        if nearest is not None and nearest[0] <= 12:
            _, which, index = nearest
            self.active_curve = which
            self.active_var.set(which)
            self.dragging = (which, index)
            self.update_status()
            self.redraw()
            return

        # If no point was selected, a click inside the plot adds a new control point.
        if self.inside_plot(event.x, event.y):
            world_x, world_y = self.screen_to_world(event.x, event.y)
            world_x = float(np.clip(world_x, self.WORLD_X_MIN, self.WORLD_X_MAX))
            world_y = float(np.clip(world_y, self.WORLD_Y_MIN, self.WORLD_Y_MAX))
            self.controls[self.active_curve].append([world_x, world_y])
            self.redraw()

    def on_canvas_drag(self, event) -> None:
        """Move the selected control point while the left mouse button is held."""
        if self.dragging is None:
            return

        which, index = self.dragging
        world_x, world_y = self.screen_to_world(event.x, event.y)
        world_x = float(np.clip(world_x, self.WORLD_X_MIN, self.WORLD_X_MAX))
        world_y = float(np.clip(world_y, self.WORLD_Y_MIN, self.WORLD_Y_MAX))
        self.controls[which][index] = [world_x, world_y]
        self.redraw()

    def on_canvas_release(self, _event) -> None:
        self.dragging = None

    def undo_point(self) -> None:
        if self.controls[self.active_curve]:
            self.controls[self.active_curve].pop()
        self.redraw()

    def clear_active(self) -> None:
        self.controls[self.active_curve].clear()
        self.redraw()

    def reset_examples(self) -> None:
        """Restore the initial Bézier/B-spline example and reset the morph."""
        self.stop_animation()
        self.controls = {
            "A": [[0.8, 0.8], [1.0, 5.7], [3.4, 5.6], [4.4, 0.9]],
            "B": [
                [1.0, 1.1], [0.7, 3.8], [2.0, 6.2],
                [4.1, 5.1], [5.6, 1.0], [8.8, 3.3],
            ],
        }
        self.curve_types = {"A": "bezier", "B": "bspline"}
        self.type_a_var.set("bezier")
        self.type_b_var.set("bspline")
        self.active_curve = "A"
        self.active_var.set("A")
        self.progress_var.set(0.0)
        self.progress = 0.0
        self.redraw()


# -----------------------------------------------------------------------------
# 6. Morphing controls and animation
# -----------------------------------------------------------------------------

    def on_progress_change(self, value: str) -> None:
        """Update the morphing parameter when the slider is moved."""
        try:
            self.progress = float(value)
        except ValueError:
            self.progress = float(self.progress_var.get())
        self.redraw()

    def reset_progress(self) -> None:
        self.progress_var.set(0.0)
        self.progress = 0.0
        self.redraw()

    def start_if_ready(self) -> None:
        if self.root.winfo_exists() and not self.playing:
            self.start_animation()

    def toggle_play(self) -> None:
        if self.playing:
            self.stop_animation()
        else:
            self.start_animation()

    def start_animation(self) -> None:
        if len(self.controls["A"]) < 2 or len(self.controls["B"]) < 2:
            messagebox.showinfo(
                "More control points needed",
                "Add at least 2 control points to each curve before playing the morph.",
            )
            return

        self.playing = True
        self.play_button.configure(text="Pause")
        self.animation_tick()

    def stop_animation(self) -> None:
        self.playing = False
        if hasattr(self, "play_button"):
            self.play_button.configure(text="Play")

        if self.after_id is not None:
            try:
                self.root.after_cancel(self.after_id)
            except tk.TclError:
                pass
            self.after_id = None

    def animation_tick(self) -> None:
        """Advance the morph parameter and schedule the next animation frame."""
        if not self.playing:
            return

        next_progress = self.progress + ANIMATION_STEP
        if next_progress > 1.0:
            next_progress = 0.0

        self.progress_var.set(next_progress)
        self.progress = next_progress
        self.redraw()

        # Tkinter calls this method again after the specified delay.
        self.after_id = self.root.after(ANIMATION_DELAY_MS, self.animation_tick)

    def close(self) -> None:
        self.stop_animation()
        self.root.destroy()


## 4. Program entry point

In [8]:
def main() -> None:
    """Create the main window and start Tkinter's event loop."""
    try:
        root = tk.Tk()
    except tk.TclError as exc:
        raise SystemExit(
            "Could not open the desktop window. Run this script in a local desktop session, "
            "not in a headless environment. Original error: " + str(exc)
        ) from exc

    CurveMorphingApp(root)
    root.mainloop()

main()
